In [3]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_community.document_loaders import DirectoryLoader

C:\Users\Admin\AppData\Local\Temp\ipykernel_22688\3109524755.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [4]:
dir_loader = DirectoryLoader(
    "./data", glob="**/*.pdf", loader_cls=PyMuPDFLoader,show_progress=True
)

documents = dir_loader.load()

100%|██████████| 5/5 [00:00<00:00, 11.17it/s]


In [5]:
documents

[Document(metadata={'producer': 'PDFium', 'creator': 'Microsoft Word', 'creationdate': '2025-08-13T12:44:32+00:00', 'source': 'data\\Energiemanagementsystem_ISO_50001_2018_DESTU_000589_EN.pdf', 'file_path': 'data\\Energiemanagementsystem_ISO_50001_2018_DESTU_000589_EN.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': 'GenericCertificate_50001_NewTemplate', 'author': '', 'subject': 'GenericCertificate_50001_NewTemplate', 'keywords': '', 'moddate': '2025-08-13T12:44:32+00:00', 'trapped': '', 'encryption': 'Standard V2 R3 128-bit RC4', 'modDate': "D:20250813124432+00'00'", 'creationDate': "D:20250813124432+00'00'", 'page': 0}, page_content='Certificate DE25/00000589 \nThe Energy management system of \nRASTAL GMBH & CO. KG \nRastal-Straße 1, DE 56203 Höhr-Grenzhausen \nhas been assessed and certified as meeting the requirements of \nISO 50001:2018 \nFor the following activities \nDesign, technology, innovation in the production and finishing of glass and ceramics \nThis certificate is 

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
split_docs = text_splitter.split_documents(documents)
print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

Split 18 documents into 75 chunks


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedder = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

vector_db = Chroma(collection_name='my_collection',
                   embedding_function=embedder,
                   persist_directory='./chroma_langchain_db')

c:\Users\Admin\Documents\GitHub\potens-intern-AI\ML-chaitanya-powar\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3236.39it/s]


In [7]:
vector_db.add_documents(documents)

['47d521a8-6755-4387-b617-1ae3c263b333',
 '9e4a5ea6-ef98-402a-81f8-6fdd35dc4b55',
 '45682c40-da23-43fd-b8d0-f9344edcda86',
 'd155ba0b-2baf-4f24-a41e-bbb70b2fbe15',
 '045b2f3f-b777-4b91-bb73-f1c00b6bbce7',
 'd42d90ff-89a0-4684-9141-22a00219f3de',
 'ed59bf64-5e40-49dc-a9df-f0960c0110e8',
 'c2b6da4e-ef84-449f-8575-4f382a51623c',
 'd32d9b3b-e68d-4c68-bd33-e81f6695274f',
 'd16ba9b2-ea17-42d1-af0b-55a8fd9b8baa',
 '18f4037c-d0da-46f4-8bf3-c0608e4f72e5',
 '4ea54b07-bef8-48be-8308-8dbd54539262',
 'af73fcf7-d60c-428e-89df-12d369827a6b',
 '5546328d-2034-475f-822d-203ed3fd49cf',
 '81cb016a-e017-42cf-941d-d9a9533f02f4',
 '8138df0c-fa26-4c8f-81c8-d181f5140ab5',
 'e557cb5c-1684-4153-adb0-c5a382b18c97',
 '423c3b08-15ea-4b3f-aca8-56fb2d009a97']

In [10]:
results = vector_db.similarity_search_with_score(
    "who is responsible for delivery?", k=4
)
print(results)
context = []


[(Document(id='a51d4239-b18f-49dd-ba8e-295724713546', metadata={'moddate': '2019-09-10T12:43:33+02:00', 'file_path': 'data\\Rastal_General_Terms_and_Conditions_of_Business.pdf', 'creator': 'Microsoft® Word 2010', 'creationDate': "D:20190910124333+02'00'", 'creationdate': '2019-09-10T12:43:33+02:00', 'author': 'André Klaus', 'producer': 'Microsoft® Word 2010', 'page': 4, 'keywords': '', 'modDate': "D:20190910124333+02'00'", 'subject': '', 'trapped': '', 'title': '', 'source': 'data\\Rastal_General_Terms_and_Conditions_of_Business.pdf', 'total_pages': 10, 'format': 'PDF 1.5'}, page_content='5 \n \nnature, the goods delivery and service performance periods will be extended, and \nthe goods delivery and service performance dates will be postponed, by the period \nof time during which the impediment subsists, plus a reasonable recovery lead time. \nIf, due to the delay, the purchaser cannot be reasonably expected to take delivery \nof the goods or services, the purchaser is entitled to with

In [20]:
context = []

for i, (doc, score) in enumerate(results, start=1):
    if score > 0.9:
        context.append({
            "source_id": i,
            "content": doc.page_content,
            "metadata": doc.metadata
        })
    else:
        context.append({
            "source_id": i,
            "content": 'no appropriate context found',
            "metadata": 'no metadata found'
        })
    
print(context)

[{'source_id': 1, 'content': '5 \n \nnature, the goods delivery and service performance periods will be extended, and \nthe goods delivery and service performance dates will be postponed, by the period \nof time during which the impediment subsists, plus a reasonable recovery lead time. \nIf, due to the delay, the purchaser cannot be reasonably expected to take delivery \nof the goods or services, the purchaser is entitled to withdraw from the contract by \nsubmitting to us a written statement to this effect without delay. \n6.4 \nWe are entitled to perform partial deliveries on condition that the partial delivery is \nusable by the purchaser within the scope of the contractually intended purpose, \nthat the supply of the outstanding goods is guaranteed and that the purchaser is not \nrequired to incur substantially higher expenses or additional costs, unless we agree to \nassume these expenses or costs. \n6.5 \nIf, for any reason whatsoever, we should come into delay with a delivery o

In [21]:
from langchain_groq import ChatGroq
import os 
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GROQ_API_KEY", "").strip()
llm = None
tools = None
startup_error = None

print("api_key:", repr(api_key))
print("startup tools:", tools if "tools" in globals() else "not defined")
if api_key:
    try:
        print("Creating ChatGroq...")
        llm = ChatGroq(
            model="llama-3.3-70b-versatile",
            groq_api_key=api_key,
        )
        print("ChatGroq created")
    except Exception as exc:
        startup_error = str(exc)
else:
    startup_error = (
        "GROQ_API_KEY is not set. Add it to your environment or a .env file "
        "before calling /ask."
    )

api_key: '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'
startup tools: None
Creating ChatGroq...
ChatGroq created


In [34]:
query = "what if product comes defected"
prompt = f"""
You are a helpful document question-answering assistant.

Use ONLY the information provided in the context below.

Rules:
1. Answer only from the provided context.
2. Do not use outside knowledge.
3. If the answer is not present in the context, reply exactly:
   "I couldn't find information related to this question in the uploaded documents."
4. Cite the source number(s) used for each factual statement.

Context:
{context}

Question:
{query}

Answer:
"""

In [35]:
generalized_answer = llm.invoke(prompt)
print(generalized_answer)

content='According to the context, if the product comes defected, it is considered a defect ("Mangel") capable of triggering warranty claims [Source: 3, §11]. The warranty period is six (6) months from the date of risk transfer [Source: 1, 8.1].' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 5602, 'total_tokens': 5664, 'completion_time': 0.286453377, 'completion_tokens_details': None, 'prompt_time': 0.30471536, 'prompt_tokens_details': None, 'queue_time': 0.162418943, 'total_time': 0.591168737}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f3bf9-d4c6-7433-bc80-829a925d4d41-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 5602, 'output_tokens': 62, 'total_tokens': 5664}
